# Project Forge — ComfyUI Remote Server

Este notebook transforma o Google Colab em um servidor ComfyUI gratuito com GPU.

**GPU:** T4 (12GB VRAM) — fornecida gratuitamente pelo Google.

---

## Como usar:
1. Menu `Runtime` → `Change runtime type` → `T4 GPU`
2. Execute cada célula em ordem
3. Quando a URL do túnel aparecer, copie (Ctrl+C) — o Forge detecta automaticamente

---

In [ ]:
# @title 1. Verificar GPU
import torch, psutil, platform, subprocess, os, json, time, threading, urllib.request

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    total_mem = getattr(gpu, 'total_memory', getattr(gpu, 'total_mem', 0))
    print(f"VRAM: {round(total_mem / 1024**3, 1)} GB")
else:
    print("❌ GPU não detectada. Vá em Runtime > Change runtime type > T4 GPU")
    raise SystemExit()

In [ ]:
# @title 2. Instalar dependências do sistema
!apt-get update -qq -y
!apt-get install -qq -y git wget unzip zip libgl1-mesa-glx libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 > /dev/null 2>&1

# Instalar cloudflared (túnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas")

In [ ]:
# @title 3. Baixar/Instalar ComfyUI
COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("Baixando ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY_DIR}" 2>&1 | tail -1
else:
    print("ComfyUI já existe")

# Instalar requirements
print("Instalando dependências Python...")
!pip install -q -r "{COMFY_DIR}/requirements.txt" 2>&1 | tail -1

print("✅ ComfyUI pronto")

In [ ]:
# @title 4. Baixar modelo SD1.5
MODEL_NAME = "dreamshaper_8.safetensors"
MODEL_PATH = f"{COMFY_DIR}/models/checkpoints/{MODEL_NAME}"

if not os.path.exists(MODEL_PATH):
    print("Baixando Dreamshaper 8 (~2GB)...")
    urls = [
        "https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor&size=pruned&fp=fp16",
        "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
    ]
    ok = False
    for url in urls:
        try:
            !wget -q --show-progress -O "{MODEL_PATH}" "{url}"
            if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1_000_000_000:
                ok = True
                break
            else:
                print("Falhou, tentando outra fonte...")
        except Exception as e:
            print(f"Erro na fonte {url}: {e}")
    if not ok:
        print("❌ Nao foi possivel baixar o modelo.")
        raise SystemExit()
    print("✅ Modelo baixado")
else:
    print("✅ Modelo já existe")

print(f"\nModelo: {MODEL_NAME}")
print(f"Tamanho: {round(os.path.getsize(MODEL_PATH) / 1024**3, 1)} GB")

In [ ]:
# @title 5. (Opcional) Baixar VAE e LoRA extras
# Descomente se quiser modelos extras

# VAE melhorado
# vae_url = "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors"
# !wget -q --show-progress -O "{COMFY_DIR}/models/vae/vae-ft-mse.safetensors" "{vae_url}"

# ControlNet (opcional)
# !wget -q --show-progress ...

In [ ]:
# @title 6. Iniciar servidor ComfyUI
import subprocess, sys

server_port = 8188

log_file = open('/content/comfyui.log', 'w')

server = subprocess.Popen(
    [sys.executable, "main.py", f"--port={server_port}", "--listen=0.0.0.0"],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"✅ ComfyUI iniciado (PID: {server.pid})")
print("Aguardando servidor ficar pronto...")

# Aguardar servidor ficar acessível
import time as tmod
for i in range(120):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        urllib.request.urlopen(req, timeout=2)
        print(f"✅ Servidor pronto após {i+1}s")
        break
    except:
        tmod.sleep(1)
else:
    print("❌ Servidor não iniciou. Verifique os logs em /content/comfyui.log")
    server.terminate()
    raise SystemExit()

In [ ]:
# @title 7. Iniciar túnel público (Cloudflare -> SSH -> LocalTunnel)
import subprocess, threading, time

tunnel_url = None

def find_url(text):
    for word in text.split():
        w = word.rstrip(".,;\"'")
        if "https://" in w and (
            ".trycloudflare.com" in w
            or ".loca.lt" in w
            or ".localhost.run" in w
            or "serveo.net" in w
        ):
            return w
    return None

def try_cloudflared():
    global tunnel_url
    try:
        proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        for line in proc.stdout:
            line = line.strip()
            if "trycloudflare.com" in line:
                u = find_url(line)
                if u:
                    tunnel_url = u
                    return
            time.sleep(0.1)
    except Exception as e:
        print(f"[cloudflared] erro: {e}")

def try_ssh():
    global tunnel_url
    try:
        proc = subprocess.Popen(
            ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30",
             "-R", "80:localhost:8188", "nokey@localhost.run"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
        )
        for line in proc.stdout:
            line = line.strip()
            if "localhost.run" in line:
                u = find_url(line)
                if u:
                    tunnel_url = u
                    return
            time.sleep(0.1)
    except Exception as e:
        print(f"[ssh] erro: {e}")

print("Tentando Cloudflare...")
t1 = threading.Thread(target=try_cloudflared, daemon=True)
t1.start()
for _ in range(90):
    if tunnel_url:
        break
    time.sleep(1)

if not tunnel_url:
    print("Cloudflare falhou. Tentando via SSH (localhost.run)...")
    t2 = threading.Thread(target=try_ssh, daemon=True)
    t2.start()
    for _ in range(60):
        if tunnel_url:
            break
        time.sleep(1)

if not tunnel_url:
    print("SSH falhou. Tentando LocalTunnel (ultimo recurso)...")
    !npm install -g -q localtunnel 2>/dev/null
    !npx localtunnel --port 8188 &
    time.sleep(8)

if tunnel_url:
    print("")
    print("=" * 62)
    print("URL DO TUNEL: " + tunnel_url)
    print("=" * 62)
else:
    print("")
    print("NAO foi possivel criar o tunel automaticamente.")
    print("Rode a proxima celula e siga as instrucoes.")


In [ ]:
# @title 8. Status do tunel (diagnostico)
print("Tunel atual:", tunnel_url)
if not tunnel_url:
    print("""
    Sem tunel ativo. O que fazer:
    1) Clique na celula 7 e aperte SHIFT+ENTER para tentar de novo.
    2) Confirme que a celula 2 instalou o cloudflared sem erro.
    3) Se o LocalTunnel pedir confirmacao, abra o link .loca.lt no
       navegador e digite o codigo pedido.
    4) O Forge ainda funciona com o gerador automatico enquanto isso.
    """)
else:
    print("Tunel OK! Copie a URL acima e cole no Forge em Configuracoes.")


In [ ]:
# @title 9. Instrucoes finais
if tunnel_url:
    print(f"""
========================================
   PROJECT FORGE - COMFYUI REMOTE
========================================
   Servidor rodando com GPU T4 gratuita

   URL DO TUNEL:
   {tunnel_url}

   COMO CONECTAR NO FORGE:
   1) Copie a URL acima (Ctrl+C)
   2) No Forge, clique em Configuracoes
   3) Cole em "URL do servidor" e clique em Testar
   4) Deve aparecer CONECTADO. Pode criar!

   ATENCAO:
   - O notebook precisa ficar rodando.
   - A sessao expira em ~12h. Quando expirar,
     execute tudo de novo e cole a nova URL.
========================================
""")
else:
    print("""
   SEM TUNEL ATIVO.
   1) Rode a celula 7 novamente (clique nela e SHIFT+ENTER).
   2) Veja o diagnostico da celula 8.
""")


In [ ]:
# @title ⏳ Keep Alive (evita desconexão)
# Esta célula mantém o notebook ativo.
# Execute se for usar por mais de 30min.

import time, threading, requests

def ping_loop():
    while True:
        try:
            requests.get("https://www.google.com", timeout=10)
        except:
            pass
        time.sleep(60)

if tunnel_url:
    t = threading.Thread(target=ping_loop, daemon=True)
    t.start()
    print("✅ Keep alive ativo (ping a cada 60s)")
else:
    print("Configure o túnel primeiro")